In [ ]:
#import libraries
import numpy as np
import pandas as pd

In [ ]:
# load the data to explore
data = pd.read_csv("Store.csv", encoding='latin-1')
data

In [ ]:
data.shape

In [ ]:
data.head(2)

In [ ]:
data.tail(2)

In [ ]:
data.columns

In [ ]:
data.dtypes

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated()

In [ ]:
data.drop_duplicates()

In [ ]:
#Splitting columns based on data types 
num_col = data.select_dtypes(include=np.number).columns
num_col

In [ ]:
tex_col = data.select_dtypes(include = object).columns
tex_col

In [ ]:
# Fix both date columns
data['Order Date'] = pd.to_datetime(data['Order Date'])
data['Ship Date']  = pd.to_datetime(data['Ship Date'])

# Verify the fix
print(data[['Order Date', 'Ship Date']].dtypes)

In [ ]:
data.dtypes

In [ ]:
def check_skew(data, col): 
    mean = data[col].mean() 
    median = data[col].median() 
    ratio = mean / median 
    skew = data[col].skew() 
    
    print(f"Column : {col}") 
    print(f"Mean : {mean:.2f}") 
    print(f"Median : {median:.2f}") 
    print(f"Ratio : {ratio:.2f}x") 
    print(f"Skew : {skew:.2f}") 
    
    if ratio > 2: 
        print("⚠ HIGH SKEW — investigate outliers") 
    else: 
        print("✓ No major skew") 

# Run on all numeric columns at once 
for col in ['Sales', 'Profit', 'Quantity', 'Discount']: 
    check_skew(data, col) 
    print("---")

In [ ]:
# IQR method — works for any skewed column
# Run this for Sales (then repeat for Profit)

col = 'Sales'

Q1    = data[col].quantile(0.25)
Q3    = data[col].quantile(0.75)
IQR   = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1          : {Q1:.2f}")
print(f"Q3          : {Q3:.2f}")
print(f"IQR         : {IQR:.2f}")
print(f"Lower fence : {lower:.2f}")
print(f"Upper fence : {upper:.2f}")
print(f"Outlier rows: {len(data[data[col] > upper])}")

In [ ]:
# IQR method — works for any skewed column
# Run this for Sales (then repeat for Profit)

col = 'Profit'

Q1    = data[col].quantile(0.25)
Q3    = data[col].quantile(0.75)
IQR   = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1          : {Q1:.2f}")
print(f"Q3          : {Q3:.2f}")
print(f"IQR         : {IQR:.2f}")
print(f"Lower fence : {lower:.2f}")
print(f"Upper fence : {upper:.2f}")
print(f"Outlier rows: {len(data[data[col] > upper])}")

In [ ]:
#look at outliers before deciding what to do
data[data['Sales'] > upper][[
    'Category', 'Sub-Category',
    'Sales', 'Quantity', 'Segment'
]].sort_values('Sales', ascending=False).head(8)

In [ ]:
data[data['Profit'] > upper][[
    'Category', 'Sub-Category',
    'Sales', 'Quantity', 'Segment'
]].sort_values('Sales', ascending=False).head(8)

In [ ]:
data['is_high_value'] = data['Sales'] > upper

# Do the same for Profit losses
data['profit_status'] = data['Profit'].apply(
    lambda x: 'Loss' if x < 0 else 'Profit'
)

print(data['is_high_value'].value_counts())
print()
print(data['profit_status'].value_counts())

In [ ]:
# Now that they're dates, we can do math!
data['shipping_days'] = (data['Ship Date'] - data['Order Date']).dt.days

print(data['shipping_days'].describe())

In [ ]:
# Confirm the new columns were added
print("Shape:", data.shape)
print("New columns:", ['shipping_days', 'is_high_value', 'profit_status'])

# Save clean file for Power BI
data.to_csv('superstore_clean.csv', index=False)
print("\n✓ Saved as superstore_clean.csv")
print(f"✓ Total columns: {data.shape[1]}") 

In [ ]:
data